# Part 4 — LLM-Powered Feature: Tabular Record Batch Scoring (Track B)
### Loan Approval / Credit Risk Dataset

**Chosen track: (B) Tabular Record Batch Scoring.** Three real applicant records from `cleaned_data.csv` are
each scored against a lending-risk rubric via an LLM call, with the response validated against a JSON schema.

**Run this notebook in Google Colab.** The first code cell installs dependencies; the second securely prompts
for your API key with `getpass` (it is never hardcoded, never printed, and never saved to any file — it only
lives in the `LLM_API_KEY` environment variable for the duration of this Colab session).

This notebook uses **OpenRouter** (`https://openrouter.ai/api/v1/chat/completions`), which accepts any
OpenAI-compatible key and lets you pick from free-tier models — get a key at
[openrouter.ai/keys](https://openrouter.ai/keys) if you don't have one yet.


## Setup

In [ ]:
!pip install -q requests jsonschema pandas


In [ ]:
import os
import re
import json
import time
import getpass
import requests
import pandas as pd
from jsonschema import validate, ValidationError

# Securely prompt for the key -- this is the ONLY place it is entered.
# It is stored in an environment variable for this session only; never written to disk.
os.environ['LLM_API_KEY'] = getpass.getpass('Paste your OpenRouter API key (input hidden): ')
print("Key loaded into environment variable LLM_API_KEY (not printed, not saved).")


## Load Three Records to Score

If you're running this in Colab, upload `cleaned_data.csv` from Part 1 first (Files pane -> upload), or replace
the path below with wherever you've placed it (e.g. mounted Google Drive).


In [ ]:
# Upload cleaned_data.csv in the Colab file browser first, then run this cell.
df = pd.read_csv('cleaned_data.csv')

# Three real, diverse applicant profiles picked from the cleaned dataset
record_ids = ['LP001066', 'LP001006', 'LP001014']  # low / mid / high credit-risk profiles
records = df[df['Loan_ID'].isin(record_ids)].to_dict('records')

for r in records:
    print(json.dumps(r, indent=2))
    print()


## The `call_llm` Function

In [ ]:
LLM_MODEL = "meta-llama/llama-3.1-8b-instruct:free"  # swap for any OpenRouter model you prefer
LLM_URL = "https://openrouter.ai/api/v1/chat/completions"

def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    """Calls the LLM API and returns the response text, or None on failure."""
    api_key = os.environ.get('LLM_API_KEY')
    if not api_key:
        print("No API key found in LLM_API_KEY.")
        return None

    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    response = requests.post(LLM_URL, headers=headers, json=payload)

    if response.status_code != 200:
        print(f"API call failed: {response.status_code}")
        print(response.text[:300])
        return None

    return response.json()['choices'][0]['message']['content']


In [ ]:
# Demonstrate the function with a simple test prompt
test_output = call_llm(
    system_prompt="You are a helpful assistant.",
    user_prompt="Reply with only the word: hello",
    temperature=0.0
)
print("Test output:", test_output)


## Prompt Design

**System prompt** (states the LLM's role, the scoring rubric, and includes one worked example):


In [ ]:
SYSTEM_PROMPT = """You are a loan-risk scoring assistant for a lending institution. You will be given a
single loan applicant record as a JSON object. Score the applicant against the following rubric and respond
with ONLY a valid JSON object -- no markdown, no explanation, no code fences.

RUBRIC:
- risk_tier: "low" if Credit_History == 1 AND ApplicantIncome is comfortably above LoanAmount-implied
  repayment needs (roughly ApplicantIncome > 4x LoanAmount / (Loan_Amount_Term/12)); "medium" if
  Credit_History == 1 but income is only marginally sufficient, or Credit_History is missing; "high" if
  Credit_History == 0, regardless of income.
- flag_for_review: true if Credit_History == 0, OR if ApplicantIncome and CoapplicantIncome combined seem
  inconsistent with the requested LoanAmount (very high loan relative to combined income); false otherwise.
- primary_signal: the single field that most influenced your risk_tier decision (e.g. "Credit_History",
  "ApplicantIncome", "income-to-loan ratio").
- confidence: "low", "medium", or "high" -- how confident you are in this assessment given the fields
  available.
- recommended_action: one short sentence describing the suggested next step for a loan officer.

Required JSON fields: risk_tier, flag_for_review, primary_signal, confidence, recommended_action.

WORKED EXAMPLE:
Input: {"Loan_ID": "LPEXAMPLE", "ApplicantIncome": 2000, "CoapplicantIncome": 0, "LoanAmount": 250,
"Loan_Amount_Term": 360, "Credit_History": 0.0, "Dependents": 2, "Property_Area": "Rural"}
Output: {"risk_tier": "high", "flag_for_review": true, "primary_signal": "Credit_History",
"confidence": "high", "recommended_action": "Escalate to a senior underwriter before proceeding; credit
history does not meet guidelines."}

Now score the applicant record provided in the user message. Respond with ONLY the JSON object."""


**User prompt template** (the record is inserted as a JSON string):

```
json.dumps(record, indent=2)
```

**Why temperature=0**: this is a structured scoring task where we want the same input to reliably produce the
same output every time it's run -- consistent, reproducible batch scoring is more valuable here than creative
variation. A temperature near 0 makes the model consistently pick its highest-probability next token, which is
exactly the deterministic behavior a structured-output pipeline needs.


## Temperature A/B Comparison

For each of the three records, we call the LLM twice -- once at `temperature=0` and once at `temperature=0.7`
-- and compare.


In [ ]:
temp_comparison_rows = []

for record in records:
    user_prompt = json.dumps(record, indent=2)

    output_t0 = call_llm(SYSTEM_PROMPT, user_prompt, temperature=0.0)
    time.sleep(1)
    output_t07 = call_llm(SYSTEM_PROMPT, user_prompt, temperature=0.7)
    time.sleep(1)

    temp_comparison_rows.append({
        'Loan_ID': record['Loan_ID'],
        'Output (temp=0)': output_t0,
        'Output (temp=0.7)': output_t07
    })

temp_comparison_df = pd.DataFrame(temp_comparison_rows)
for _, row in temp_comparison_df.iterrows():
    print(f"--- {row['Loan_ID']} ---")
    print("temp=0.0:", row['Output (temp=0)'])
    print("temp=0.7:", row['Output (temp=0.7)'])
    print()


Fill in the **Key difference** column after inspecting the printed outputs above, then record all three
rows in your README as a table with columns: Input, Output at temp=0, Output at temp=0.7, Key difference.

**Why temperature=0 is more deterministic**: the model always selects the single highest-probability next
token at each step, so the same prompt reliably reproduces the same (or near-identical) output. **Why
temperature=0.7 introduces variability**: the model instead samples from a broader slice of the probability
distribution over next tokens, so plausible-but-different phrasings (and occasionally different risk_tier
judgment calls near a boundary) can appear across repeated calls.


## PII Guardrail

In [ ]:
def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))


def call_llm_guarded(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None
    return call_llm(system_prompt, user_prompt, temperature=temperature, max_tokens=max_tokens)


In [ ]:
# Test 1: input WITH an email address -- should be blocked
blocked_test = call_llm_guarded(
    SYSTEM_PROMPT,
    'Applicant contact: jane.doe@example.com. ' + json.dumps(records[0])
)
print("Blocked test result:", blocked_test)

print()

# Test 2: clean input, no PII -- should proceed to the LLM call
clean_test = call_llm_guarded(SYSTEM_PROMPT, json.dumps(records[0]))
print("Clean test result:", clean_test)


## Structured Output Handling: Schema Validation

In [ ]:
RISK_SCHEMA = {
    "type": "object",
    "properties": {
        "risk_tier": {"type": "string", "enum": ["low", "medium", "high"]},
        "flag_for_review": {"type": "boolean"},
        "primary_signal": {"type": "string"},
        "confidence": {"type": "string", "enum": ["low", "medium", "high"]},
        "recommended_action": {"type": "string"}
    },
    "required": ["risk_tier", "flag_for_review", "primary_signal", "confidence", "recommended_action"]
}

FALLBACK = {
    "risk_tier": None, "flag_for_review": None, "primary_signal": None,
    "confidence": None, "recommended_action": None
}

def score_record(record):
    user_prompt = json.dumps(record, indent=2)
    raw_response = call_llm_guarded(SYSTEM_PROMPT, user_prompt, temperature=0.0)

    if raw_response is None:
        return FALLBACK, "blocked_or_failed", raw_response

    try:
        parsed = json.loads(raw_response.strip())
    except json.JSONDecodeError as e:
        print(f"JSON parse error for {record.get('Loan_ID')}: {e}")
        return FALLBACK, f"fail: JSONDecodeError - {e}", raw_response

    try:
        validate(instance=parsed, schema=RISK_SCHEMA)
    except ValidationError as e:
        print(f"Schema validation error for {record.get('Loan_ID')}: {e.message}")
        return FALLBACK, f"fail: ValidationError - {e.message}", raw_response

    return parsed, "pass", raw_response


## End-to-End Demonstration on Three Records

In [ ]:
demo_rows = []

for record in records:
    assessment, status, raw = score_record(record)
    demo_rows.append({
        'Loan_ID': record['Loan_ID'],
        'Input Record': json.dumps(record),
        'LLM Raw Response': raw,
        'Assessment': json.dumps(assessment),
        'Validation Status': status
    })
    time.sleep(1)

demo_df = pd.DataFrame(demo_rows)
for _, row in demo_df.iterrows():
    print(f"--- {row['Loan_ID']} ---")
    print("Input:", row['Input Record'])
    print("Raw response:", row['LLM Raw Response'])
    print("Validation:", row['Validation Status'])
    print()

demo_df[['Loan_ID', 'Assessment', 'Validation Status']]


Copy the printed table above into your README as the required 3-row demonstration table, with columns:
**Input Record | LLM Assessment JSON | Validation Status (pass/fail + error if fail)**.
